# TripAdvisor Sentiment Analysis — Step 3: Sentiment Lexicon Scoring

Welcome to the third stage of your research project! In this notebook, we will apply **Lexicon Models** to identify emotional meaning in words and calculate sentiment scores for each guest review.

### Learning Objectives:
1. Understand what a **Sentiment Lexicon** is.
2. Compare different lexicons (**AFINN**, **NRC**, and **Syuzhet**).
3. Compute word-level and review-level sentiment scores.
4. Classify reviews into Positive, Negative, and Neutral groups.

---

### ☁️ Google Colab Setup
If you are running this notebook in **Google Colab**, simply run the code cell below to download the dataset and install any missing tools. If you are running this locally in RStudio, you can skip it!

In [ ]:
# =====================================================================
# GOOGLE COLAB SETUP (Run this cell only if you are using Google Colab)
# =====================================================================
if (dir.exists("/content")) {
  if (!dir.exists("/content/MrKadekProject")) {
    cat("Downloading project files from GitHub...\n")
    system("git clone https://github.com/divanahadyan1618/MrKadekProject.git /content/MrKadekProject", ignore.stdout=TRUE, ignore.stderr=TRUE)
  }
  setwd("/content/MrKadekProject")
  
  # Install required packages if missing
  packages <- c("rvest", "tidytext", "stopwords", "syuzhet", "wordcloud", "RColorBrewer")
  new_packages <- packages[!(packages %in% installed.packages()[,"Package"])]
  if(length(new_packages)) install.packages(new_packages, quiet = TRUE)
  
  cat("Colab Environment Ready!\n")
}


## 1. Sentiment Lexicon Scoring Concept

A **Sentiment Lexicon** is a dictionary of words tagged with emotional meaning or numerical polarity scores.

**Examples:**
* `"excellent"` ➔ Positive (+4 in AFINN)
* `"dirty"` ➔ Negative (-3 in AFINN)
* `"slow"` ➔ Negative (-2 in AFINN)

The total sentiment score of a review is calculated by summing the scores of all its sentiment-carrying words.

Let's load the libraries. We will use the `syuzhet` package, which is the gold-standard in R for computing sentiment scores using multiple dictionary methods easily.

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 3: Sentiment Lexicon Scoring
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# How does a computer know if a review is happy or angry? It uses a "Lexicon".
# A Lexicon is simply a giant dictionary where scientists have manually assigned 
# numeric scores to words. For example: "excellent" = +4, "dirty" = -3.
# We will scan our clean reviews against these dictionaries to calculate a final score.

## 2. Load Cleaned Data

We read the cleaned full-sentence reviews dataframe from Step 2 to evaluate the context of each review.

In [ ]:
# =====================================================================
# STEP 1: Load Required Packages
# =====================================================================
# 'tidyverse' and 'tidytext' help us read and manipulate our tables.
library(tidyverse)
library(tidytext)

# 'syuzhet' is a very famous toolset built specifically for Sentiment Analysis.
# It contains the massive dictionaries we need to score emotions.
library(syuzhet)

cat("Libraries loaded successfully!\n")

## 3. Scoring Sentiment using the Syuzhet Package

The `syuzhet` package provides simple functions to score text using several lexicons:
* **Syuzhet Lexicon**: Developed by the NLP laboratory at Nebraska-Lincoln (default NLP method).
* **AFINN Lexicon**: Scores words on an integer scale from -5 (very negative) to +5 (very positive).

Let's calculate numeric sentiment scores for each review using these methods.

In [ ]:
# =====================================================================
# STEP 2: Open the Cleaned Reviews
# =====================================================================
# We load the full cleaned sentences from Step 2 into a box named 'cleaned_reviews'.
cleaned_reviews <- read_csv2("data/cleaned/bvlgari_cleaned_reviews.csv")

cat("Loaded", nrow(cleaned_reviews), "cleaned reviews for emotional scoring.\n")

## 4. Classify Sentiment Categories

We logically bucket the numeric `score_syuzhet` value into distinct nominal categories:
* **Positive**: score > 0
* **Neutral**: score == 0
* **Negative**: score < 0

We then print out the percentage distribution to see how guests feel overall.

In [ ]:
# =====================================================================
# STEP 3: Apply the Dictionary Lexicons (Syuzhet & AFINN)
# =====================================================================
# We are asking the computer to read every review and calculate a math score.
# We use two different methods to be thorough:
# 1. "syuzhet" method: A standard scientific scoring system.
# 2. "afinn" method: Gives an integer score from -5 (furious) to +5 (thrilled).

# 'mutate' means "create a new column in our table".
sentiment_results <- cleaned_reviews %>%
  mutate(
    score_syuzhet = get_sentiment(cleaned_text, method = "syuzhet"),
    score_afinn = get_sentiment(cleaned_text, method = "afinn")
  )

## 5. Rich Emotion Extraction (NRC Lexicon)

The **NRC Emotion Lexicon** is a list of words that associates text with eight basic human emotions: anger, anticipation, disgust, fear, joy, sadness, surprise, and trust.

**Understanding NRC Scores:** The numerical scores produced by the NRC lexicon represent the *frequency* or *count* of words associated with each specific emotion category found in the text. For example, a joy score of 3 means there were 3 words in the review associated with the emotion of joy.

Let's extract detailed emotion scores for Bvlgari Resort Bali reviews. *Note: This takes a little longer to execute as it checks against a massive dictionary.*

In [ ]:
# =====================================================================
# STEP 4: Classify the Scores into Simple Categories
# =====================================================================
# A numeric score like "1.45" is hard to understand.
# We use 'case_when' (which is like a giant IF statement) to categorize them:
# IF the score is greater than 0  -> "Positive"
# IF the score is less than 0     -> "Negative"
# IF the score is exactly 0       -> "Neutral"

sentiment_classified <- sentiment_results %>%
  mutate(
    sentiment_category = case_when(
      score_syuzhet > 0 ~ "Positive",
      score_syuzhet < 0 ~ "Negative",
      TRUE              ~ "Neutral" # TRUE here means "everything else"
    )
  )

# Now, let's group all the "Positive" and "Negative" rows together and count them!
sentiment_summary <- sentiment_classified %>%
  group_by(sentiment_category) %>%
  summarise(
    count = n(), # n() counts the number of rows
    percentage = (n() / nrow(sentiment_classified)) * 100 # Calculate the percentage %
  )

# Print the final report to the screen
print(sentiment_summary)

## 6. Save Scored Data

We save our final, heavily annotated dataset to `data/cleaned/bvlgari_sentiment_scores.csv` so it is ready for Step 4 (Data Visualization).

In [ ]:
# =====================================================================
# STEP 5: NRC Deep Emotion Extraction
# =====================================================================
# "Positive" or "Negative" is too basic. What if we want to know if guests are ANGRY or AFRAID?
# The 'NRC' lexicon checks text against 8 human emotions:
# (anger, anticipation, disgust, fear, joy, sadness, surprise, trust).

cat("Processing deep emotions. The computer is reading thousands of words, please wait a few seconds...\n")

# 'get_nrc_sentiment' returns a table showing exactly how much 'joy' or 'anger' is in the text.
nrc_emotions <- get_nrc_sentiment(sentiment_classified$cleaned_text)

# 'bind_cols' acts like glue. It takes our original table, and glues the new emotion columns to the right side of it.
final_research_data <- bind_cols(sentiment_classified, nrc_emotions)

## 🎓 Student Exercise / Assignment Connection

For your project assignment, make sure that:
1. You understand the difference between **sentiment score** (numeric scale, e.g. -2, 4) and **emotion category** (nominal category, e.g. joy, trust).
2. If you are comparing two different hotels, you should run this scoring notebook for both hotels, then compare their average scores to determine which hotel has superior customer sentiments.

**Great job! Let's open `04_visualization.ipynb` to create beautiful visuals.**